# Lab 7: Memory as a Context Provider

In this lab, you will use `Neo4jMicrosoftMemory` to give an agent persistent memory that automatically retrieves relevant past messages, entities, and preferences before each turn, and stores new ones after. Memory turns the agent from stateless to stateful — it accumulates knowledge about users and conversations as a connected graph in Neo4j.

## What you will learn

- How to configure `MemorySettings` with Neo4j connection details
- How to create `Neo4jMicrosoftMemory` from a `MemoryClient`
- How the memory context provider uses `before_run()` and `after_run()`
- How to inspect stored memories with `search_memory()`

## Setup

In [1]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

provider_name = os.getenv("LLM_PROVIDER", "openai").lower()
if provider_name == "azure":
    assert os.getenv("AZURE_OPENAI_API_KEY"), "AZURE_OPENAI_API_KEY not set in .env file"
    assert os.getenv("AZURE_OPENAI_ENDPOINT"), "AZURE_OPENAI_ENDPOINT not set in .env file"
else:
    assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not set in .env file"
assert os.getenv("NEO4J_URI"), "NEO4J_URI not set in .env file"
print("Environment loaded successfully")

Environment loaded successfully


## Import Dependencies

In [2]:
from pydantic import SecretStr

from llm_provider import get_client, get_provider

from neo4j_agent_memory import MemoryClient, MemorySettings
from neo4j_agent_memory.integrations.microsoft_agent import (
    Neo4jMicrosoftMemory,
)

## Configure Memory Settings

Create a `MemorySettings` object with your Neo4j connection details. This reuses the same `NEO4J_URI`, `NEO4J_USERNAME`, and `NEO4J_PASSWORD` environment variables from previous labs.

In [3]:
if get_provider() == "azure":
    llm_model = os.getenv("AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME", "gpt-5-mini")
else:
    llm_model = os.getenv("OPENAI_RESPONSES_MODEL_ID", "gpt-5-mini")

settings = MemorySettings(
    neo4j={
        "uri": os.environ["NEO4J_URI"],
        "username": os.environ["NEO4J_USERNAME"],
        "password": SecretStr(os.environ["NEO4J_PASSWORD"]),
    },
    extraction={
        "enable_gliner": False,  # Disabled: downloads ~500MB model from HuggingFace, impractical in a workshop
        "llm_model": llm_model,
        "llm_temperature": 1.0,  # gpt-5-mini only supports temperature=1.0
    },
)

## Connect the Memory Client

`MemoryClient` supports explicit `connect()` and `close()` methods. Using explicit calls instead of `async with` lets us split the workflow across separate notebook cells.

In [4]:
memory_client = MemoryClient(settings)
await memory_client.connect()

## Create Memory and Agent

The memory provider gives the agent three types of persistent memory: **short-term** (recent messages found by semantic search), **long-term** (entities and preferences extracted from conversations), and **reasoning** (traces of past agent executions). Create `Neo4jMicrosoftMemory` with:
- `include_short_term=True` — inject recent and semantically relevant past messages
- `include_long_term=True` — inject extracted entities and preferences
- `include_reasoning=True` — inject similar past reasoning traces
- `extract_entities=True` — extract entities from messages after each turn

Setting `extract_entities=True` runs the extraction pipeline configured in `MemorySettings` above after every agent turn. With the current settings, this uses two stages: **spaCy** for fast rule-based named entity recognition, then an **LLM fallback** for entities spaCy missed. (GLiNER, the third pipeline stage, is disabled to avoid downloading a ~500MB model during the workshop.) The next lab explores these extraction stages in detail.

Pass `memory.context_provider` to the agent's `context_providers` list.

In [5]:
session_id = "movie-chat-session"

memory = Neo4jMicrosoftMemory.from_memory_client(
    memory_client=memory_client,
    session_id=session_id,
    include_short_term=True,
    include_long_term=True,
    include_reasoning=True,
    extract_entities=True,
)

client = get_client()

agent = client.as_agent(
    name="movie-memory-agent",
    instructions=(
        "You are a movie recommendation assistant with persistent "
        "memory. You remember what users have told you about their "
        "preferences, which movies you have discussed, and what "
        "you recommended in past sessions."
    ),
    context_providers=[memory.context_provider],
)

## Run a Multi-Turn Conversation

Test the memory provider with a 3-turn conversation:

1. **Turn 1** — The user states a preference. `after_run()` saves the message and extracts entities.
2. **Turn 2** — The user asks what they said. `before_run()` retrieves the previous message and injects it.
3. **Turn 3** — The agent has both conversation history and extracted preferences available.

In [ ]:
session = agent.create_session()

queries = [
    "I really enjoy sci-fi movies, especially ones about time travel.",
    "What did I say my favorite genre was?",
    "Can you recommend something I might like?",
]

for query in queries:
    print(f"\nUser: {query}")
    response = await agent.run(query, session=session)
    print(f"Assistant: {response.text}")


User: I really enjoy sci-fi movies, especially ones about time travel.


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `created_at` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=4, column=12, offset=70>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 70, 'line': 4, 'column': 12}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (c:Conversation {session_id: $session_id})\nRETURN c\nORDER BY c.created_at DESC\nLIMIT 1\n'
Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `created_at` does not exist in database `neo4j`. Verify that the s

Assistant: Awesome — I’ll remember that you enjoy sci‑fi, especially time‑travel stories.

Here are a few time‑travel sci‑fi movies across different tones you might like (short notes to help you pick):

- Back to the Future (1985) — classic, fun, character‑driven time travel.
- Primer (2004) — low‑budget, highly technical and mind‑bending; rewards repeat watches.
- Predestination (2014) — tight paradox thriller with a major twist.
- 12 Monkeys (1995) — dark, dystopian, emotionally heavy time‑travel mystery.
- Looper (2012) — action and moral dilemmas around changing futures.
- Edge of Tomorrow (2014) — action/sci‑fi time loop with humor and strong pacing.
- Timecrimes (Los Cronocrímenes, 2007) — tense, clever Spanish thriller about loops.
- Donnie Darko (2001) — surreal, psychological, cult favorite with time elements.
- Source Code (2011) — contained, procedural approach to loops and identity.
- The Girl Who Leapt Through Time (2006) — heartfelt anime with a lighter, poignant take.

W

Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `MENTIONS` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=3, column=17, offset=94>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 94, 'line': 3, 'column': 17}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (c:Conversation {session_id: $session_id})-[:HAS_MESSAGE]->(m:Message)\nWHERE NOT (m)-[:MENTIONS]->(:Entity)\nRETURN m.id AS id, m.content AS content\nORDER BY m.timestamp ASC\n'
Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationsh

Assistant: You said you really enjoy sci‑fi movies — especially ones about time travel.

User: Can you recommend something I might like?
Assistant: Nice — since you like time‑travel sci‑fi, here are a few picks across different tones. My top single recommendation is first, then other good options:

- Predestination (2014) — top pick: a tight, character‑driven paradox thriller with a big emotional twist.  
- Primer (2004) — ultra‑technical, brain‑bending; best if you like puzzles and repeat watches.  
- 12 Monkeys (1995) — darker, more emotional and mysterious; great performances.  
- Edge of Tomorrow (2014) — action‑forward loop movie with smart pacing and fun stakes.  
- Timecrimes (Los Cronocrímenes, 2007) — tense, economical Spanish loop thriller.  
- Looper (2012) — action + moral dilemmas about fate and choice.  
- Source Code (2011) — contained, procedural loop with emotional payoff.  
- The Infinite Man (2014) — indie, comedic/romantic take on obsessive time‑looping.

Want one o

Transaction failed and will be retried in 1.0259833081969152s (Failed to read from defunct connection IPv4Address(('44.200.32.15', 7687)) (ResolvedIPv4Address(('44.200.32.15', 7687))))


## Inspect Stored Memories

After the conversation, verify what the provider stored by searching the memory directly. Messages, entities, and preferences are stored as connected nodes in Neo4j — the memory is a knowledge graph the agent builds over time.

In [7]:
results = await memory.search_memory(
    query="sci-fi movies",
    include_messages=True,
    include_entities=True,
    include_preferences=True,
    limit=5,
)

print("Messages:", results.get("messages", []))
print("Entities:", results.get("entities", []))
print("Preferences:", results.get("preferences", []))

Messages: [{'role': 'assistant', 'content': 'You said you really enjoy sci‑fi movies — especially ones about time travel.', 'id': '74debee0-556b-42be-a2a5-e884019f848a'}, {'role': 'user', 'content': 'I really enjoy sci-fi movies, especially ones about time travel.', 'id': 'e3c63fbc-a05e-4473-8a15-78278a7d5876'}, {'role': 'assistant', 'content': 'Awesome — I’ll remember that you enjoy sci‑fi, especially time‑travel stories.\n\nHere are a few time‑travel sci‑fi movies across different tones you might like (short notes to help you pick):\n\n- Back to the Future (1985) — classic, fun, character‑driven time travel.\n- Primer (2004) — low‑budget, highly technical and mind‑bending; rewards repeat watches.\n- Predestination (2014) — tight paradox thriller with a major twist.\n- 12 Monkeys (1995) — dark, dystopian, emotionally heavy time‑travel mystery.\n- Looper (2012) — action and moral dilemmas around changing futures.\n- Edge of Tomorrow (2014) — action/sci‑fi time loop with humor and stron

## Close the Connection

Always close the `MemoryClient` when you are done.

In [8]:
await memory_client.close()
print("Connection closed")

Connection closed


## Summary

`Neo4jMicrosoftMemory` provides a context provider that retrieves relevant memories in `before_run()` and stores new messages and extracted entities in `after_run()`. The agent gets persistent memory across turns and sessions without calling any tools.

**What's next:** In the next lab, you will explore the entity extraction pipeline that powers the automatic extraction you saw in this lab — learning how spaCy, GLiNER, and LLM fallback stages work together to pull structured entities from conversation text.